# Day 49: Evaluation, Overfitting and Data Leakage

**Today's question:** when should we *not* trust a model's score?

Three ways a score can lie: the model **memorised** instead of learned (overfitting), the score hides the
mistakes we care about (precision and recall), or the model **peeked** at the answer (data leakage).

Before running a cell marked 🤔, predict what it will show.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

DATA_DIR = Path("../../../datasets")
df = pd.read_csv(DATA_DIR / "students.csv")
df["has_job_num"] = (df["has_job"] == "yes").astype(int)

features = ["study_hours", "attendance_pct", "sleep_hours", "previous_score", "assignments_done", "has_job_num"]
X_train, X_test, y_train, y_test = train_test_split(df[features], df["passed"], test_size=0.2, random_state=42)

## 1. Warm-up: a tree with no limit

🤔 **Predict:** a tree with **no** `max_depth` (it may ask as many questions as it likes). Training accuracy?
Test accuracy?

In [ ]:
no_limit = DecisionTreeClassifier(random_state=42)
no_limit.fit(X_train, y_train)
print("training accuracy:", no_limit.score(X_train, y_train))
print("test accuracy:    ", round(no_limit.score(X_test, y_test), 3))
print("number of leaves: ", no_limit.get_n_leaves())

100% on the students it learned from, 73% on new ones. With 45 leaves for 238 students, many leaves
describe just one or two students: the tree has **memorised** them.

## 2. Underfitting and overfitting

Three students prepare for an exam using past papers:

| Student | How they studied | Past papers | Real exam | Name |
|---------|------------------|-------------|-----------|------|
| Omar | learned one trick: "always answer C" | poor | poor | **underfitting**: too simple |
| Sara | memorised every past answer, typos included | perfect | poor | **overfitting**: memorised, did not learn |
| Mei | understood the main ideas | good | good | **good fit** |

The warning sign of overfitting is a **big gap** between training score and test score.

## 3. The depth experiment

Train one tree for each `max_depth` from 1 to 12 and record both scores.

🤔 **Predict:** as the depth grows, what happens to the training accuracy? And to the test accuracy?

In [ ]:
rows = []
for depth in range(1, 13):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    rows.append({
        "max_depth": depth,
        "train_accuracy": round(tree.score(X_train, y_train), 3),
        "test_accuracy": round(tree.score(X_test, y_test), 3),
    })

experiment = pd.DataFrame(rows)
experiment["gap"] = (experiment["train_accuracy"] - experiment["test_accuracy"]).round(3)
experiment

In [ ]:
plt.plot(experiment["max_depth"], experiment["train_accuracy"], marker="o", label="training data")
plt.plot(experiment["max_depth"], experiment["test_accuracy"], marker="o", label="test data (new students)")
plt.axhline(0.65, color="gray", linestyle="--", label="baseline (always pass)")
plt.xlabel("max_depth (questions in a row)")
plt.ylabel("accuracy")
plt.title("Deeper trees memorise: training goes up, test does not")
plt.legend()
plt.show()

How to read it (`plt.axhline` draws a horizontal reference line):
- **Depth 1: underfitting.** Both scores are low; one question is too simple (it says "pass" to everyone).
- **Depth 2–3: the sweet spot.** Test accuracy is highest (0.767 at depth 3), the gap is small.
- **Depth 5+: overfitting.** Training climbs to 1.0, test drops back to about 0.70–0.73. The extra questions
  describe individual training students, not general patterns.

Careful: the test set has only 60 students, so one student moves the score by 0.017. Look at the overall shape,
not at single bumps.

## 4. Precision and recall: which mistakes matter?

**The story.** The school wants to **flag students who are likely to fail** so a tutor contacts them in week 3.
The class we are hunting for is **fail (0)**.

- A **missed** student (failed, not flagged) gets no help. That is the costly mistake.
- A **false alarm** (flagged, but would have passed) costs a tutor 30 minutes and a slightly worried student.

We use yesterday's logistic regression.

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
predictions = log_model.predict(X_test)
print(confusion_matrix(y_test, predictions))

| | flagged (said fail) | not flagged (said pass) |
|-|-----|-----|
| **really failed (21)** | 15 caught | 6 missed |
| **really passed (39)** | 3 false alarms | 36 fine |

Two questions the school asks:

- **Recall**: *of the students who really failed, what share did we flag?* = caught / really failed = 15 / 21
- **Precision**: *of the students we flagged, what share really failed?* = caught / all flagged = 15 / (15 + 3)

In [ ]:
caught, missed, false_alarms = 15, 6, 3

print("recall by hand:   ", round(caught / (caught + missed), 2))
print("precision by hand:", round(caught / (caught + false_alarms), 2))

scikit-learn computes the same numbers. `pos_label=0` tells it that **fail** (0) is the class we are hunting for.

In [ ]:
print("recall:   ", round(recall_score(y_test, predictions, pos_label=0), 2))
print("precision:", round(precision_score(y_test, predictions, pos_label=0), 2))

In words: **we find 71% of the students who need help** (recall), and **83% of the students we flag really need
it** (precision). For this school, recall matters more: a missed student is worse than a wasted half-hour.

Memory trick: **recall** = "did we *recall* (find) all of them?" · **precision** = "were our flags *precise*?"

## 5. (Optional) Flag more students: moving the threshold

By default a student is flagged when P(pass) < 0.5. A careful school could flag anyone below 0.7.
🤔 **Predict:** what happens to recall? To precision?

In [ ]:
pass_probability = log_model.predict_proba(X_test)[:, 1]

rows = []
for threshold in [0.5, 0.6, 0.7, 0.8]:
    flagged_predictions = (pass_probability >= threshold).astype(int)   # 0 = flagged as "will fail"
    rows.append({
        "flag if P(pass) below": threshold,
        "students flagged": (flagged_predictions == 0).sum(),
        "recall": round(recall_score(y_test, flagged_predictions, pos_label=0), 2),
        "precision": round(precision_score(y_test, flagged_predictions, pos_label=0), 2),
    })
pd.DataFrame(rows)

Flag more → find more of the students in need (recall ↑) but with more false alarms (precision ↓).
At 0.8 we find **all 21**, but 12 of the 33 flagged students would have been fine. Which threshold is right is a
**human decision** about costs (how many tutors do we have?), not a maths question.

## 6. Data leakage: the "too good to be true" model

Leakage = the model gets information during training that it would **not** have at prediction time.
Watch what happens if we add `final_score` as a feature.

In [ ]:
leaky_features = features + ["final_score"]
Xl_train, Xl_test, yl_train, yl_test = train_test_split(df[leaky_features], df["passed"], test_size=0.2, random_state=42)

leaky_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
leaky_tree.fit(Xl_train, yl_train)
print("test accuracy:", leaky_tree.score(Xl_test, yl_test))

🤔 A perfect score on new students. Is this our best model? Look inside before celebrating:

In [ ]:
print(export_text(leaky_tree, feature_names=leaky_features))

The tree asks only one question: *is the final score ≤ 50?* That is the **definition** of `passed`. It learned
nothing about students. And it is useless in real life: in week 3, when the school wants the prediction, nobody
knows the final score yet.

**Rule:** if a model looks perfect, suspect leakage **first**. For each feature ask: *"Would I know this value at
the moment I need the prediction?"*

The fix is to remove the leaking column. Our honest model (depth 3) scores 0.767, which is lower and **true**.

## 7. Two more leaks: deciding with the test set, and duplicates

**Leak A: using the test set to make decisions.** In section 3 we looked at the test scores and saw that
depth 3 was best. If we now report "0.767", the test set has helped choose the model: it is no longer a fair,
unseen exam. Honest way: split the **training** data again into a smaller training part and a **validation**
part, choose `max_depth` on validation, and use the test set **once**, at the very end.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

for depth in range(1, 7):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_fit, y_fit)
    print("depth", depth, "validation accuracy:", round(tree.score(X_val, y_val), 3))

In [ ]:
chosen = DecisionTreeClassifier(max_depth=2, random_state=42)   # best on validation
chosen.fit(X_train, y_train)
print("honest test accuracy:", round(chosen.score(X_test, y_test), 3))

The validation set picks depth 2, whose honest test score is 0.733, not the 0.767 we "found" by peeking.
Choosing the best of 12 tries on the same exam flatters the result.

**Leak B: the same row on both sides of the split.** Imagine the data file accidentally contains every student
twice. `pd.concat([df, df])` (new) stacks two tables on top of each other, so we can simulate it.

In [ ]:
doubled = pd.concat([df, df])
Xd_train, Xd_test, yd_train, yd_test = train_test_split(doubled[features], doubled["passed"], test_size=0.2, random_state=42)

memoriser = DecisionTreeClassifier(random_state=42)
memoriser.fit(Xd_train, yd_train)
print("test accuracy with duplicates:", round(memoriser.score(Xd_test, yd_test), 3))
print("without duplicates (section 1):", round(no_limit.score(X_test, y_test), 3))

Most "new" test students now have a twin in the training set, and the memorising tree recognises them. The score
jumps from 0.733 to 0.867 without the model getting any better. **Fix:** `drop_duplicates()` **before** splitting
(Day 45). Real examples: the same house listed twice, the same patient visiting three times.

Other leaks to watch for:
- a feature recorded **after** the event (`final_score`, "refund issued" when predicting complaints);
- statistics (mean, median for filling gaps) computed on **all** rows instead of the training rows only;
- an ID or date that secretly encodes the answer (e.g. IDs given in order of results).

## 8. (Optional) Cross-validation: five mini-exams

With a small dataset one test set is noisy. **Cross-validation** cuts the **training** data into 5 parts, trains 5
times (each time hiding a different part) and gives 5 scores. Their average is a steadier way to choose settings,
without touching the test set.

In [ ]:
from sklearn.model_selection import cross_val_score

for depth in [1, 2, 3, 5, 8]:
    scores = cross_val_score(DecisionTreeClassifier(max_depth=depth, random_state=42), X_train, y_train, cv=5)
    print("depth", depth, "scores:", scores.round(2), "average:", round(scores.mean(), 3))

## 9. Common mistakes

In [ ]:
# Mistake 1: our classes are the numbers 0 and 1, not the words "fail"/"pass"
recall_score(y_test, predictions, pos_label="fail")

**Fix:** `pos_label=0`. And a silent one: forgetting `pos_label` gives precision and recall for class 1 (pass),
which answers a different question ("did we find all the passers?").

In [ ]:
print("recall for PASS (default):", round(recall_score(y_test, predictions), 2))
print("recall for FAIL:          ", round(recall_score(y_test, predictions, pos_label=0), 2))

Other silent mistakes: choosing settings by test score (Leak A), splitting before removing duplicates (Leak B),
and celebrating a near-perfect score without checking features.

## Recap

1. **Underfitting:** too simple, low on training and test. **Overfitting:** high on training, much lower on test.
   A small `max_depth` keeps a tree from memorising.
2. **Recall** (for the class we hunt) = share of real cases we found. **Precision** = share of our flags that were
   right. Moving the threshold trades one for the other; people decide which matters more.
3. **Leakage** = information the model would not have at prediction time: features known only afterwards,
   choosing settings with the test set, duplicates across the split.
4. Too good to be true? Look for a leak first.

**Next time:** Project 07, the ML Predictor: the whole workflow on a dataset you choose.